# Notebook 3: Build RAG Agent with LangChain

This notebook creates a conversational AI agent using your Pinecone vector database.

**Prerequisites:**
- Completed Notebook 1 (chunks created)
- Completed Notebook 2 (Pinecone vector database ready)
- API keys in .env file

## 1. Install Required Packages

In [1]:
# Install/upgrade packages
!pip install -U langchain-community langchain-core langchain-openai langchain-pinecone python-dotenv -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
from dotenv import load_dotenv

# LangChain imports (updated for new version)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Load environment variables
load_dotenv()

print("✓ Imports successful")

✓ Imports successful


## 3. Configuration

In [3]:
# Get API keys
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')

# Configuration
INDEX_NAME = 'efficient-engineer'

# Verify
if not OPENAI_API_KEY or not PINECONE_API_KEY:
    raise ValueError("API keys not found in .env file")

print("✓ API keys loaded")
print(f"✓ Index name: {INDEX_NAME}")

✓ API keys loaded
✓ Index name: efficient-engineer


## 4. Initialize Components

In [4]:
# Initialize embeddings
embeddings = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,
    model="text-embedding-ada-002"
)

# Connect to Pinecone
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7,
    openai_api_key=OPENAI_API_KEY
)

print("✓ Vector store connected")
print("✓ Retriever created")
print("✓ LLM initialized (gpt-4o)")

✓ Vector store connected
✓ Retriever created
✓ LLM initialized (gpt-4o)


## 5. Create RAG Chain (Simple Version - No Memory)

In [5]:
# Define prompt template
template = """You are an expert engineering tutor for The Efficient Engineer YouTube channel.
Use the context below to answer the question clearly and accurately.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ Simple RAG chain created")

✓ Simple RAG chain created


## 6. Test Simple RAG Chain

In [6]:
# Test the chain
question = "What is stress in engineering?"

print(f"Question: {question}\n")
print("-" * 60)

# Get answer
answer = rag_chain.invoke(question)

print(f"Answer:\n{answer}")
print("-" * 60)

Question: What is stress in engineering?

------------------------------------------------------------
Answer:
In engineering, stress is a quantity that describes the distribution of internal forces within a body as it responds to externally applied loads. It is a measure of the internal force exerted per unit area on a material. Stress allows engineers to understand how materials will react under different loading conditions, ensuring that structures and components are designed safely and effectively. Stress is typically expressed in units of Newtons per square meter (N/m²), also known as Pascals (Pa), in the SI system, or pounds per square inch (psi) in the US customary system. In the context provided, stress is particularly important for analyzing how forces, such as tensile or compressive loads, affect materials and structures.
------------------------------------------------------------


## 7. Test with Source Documents

In [8]:
# Function to get answer with sources
def ask_with_sources(question):
    # Get relevant documents
    docs = retriever.invoke(question)    
    # Get answer
    answer = rag_chain.invoke(question)
    
    return {
        'answer': answer,
        'sources': docs
    }

# Test it
question = "Explain buckling in structural engineering"
response = ask_with_sources(question)

print(f"Question: {question}\n")
print(f"Answer:\n{response['answer']}\n")
print("-" * 60)
print(f"\nSources ({len(response['sources'])}):\n")
for i, doc in enumerate(response['sources'], 1):
    print(f"{i}. {doc.metadata['video_title']}")
    print(f"   {doc.page_content[:100]}...\n")

Question: Explain buckling in structural engineering

Answer:
Buckling is a critical failure mode in structural engineering, typically associated with slender structural members subjected to compressive forces. When a structural element, such as a column or strut, is subjected to axial compression, it can become unstable and deform laterally (i.e., sideways), even if the material itself has not yet reached its yield strength. This lateral deformation is known as buckling.

The critical load at which buckling occurs depends on several factors, including the material properties, the geometry of the member, and the conditions of support or restraint at the ends of the member. Euler's formula is commonly used to estimate the critical buckling load for a long, slender, and perfectly straight column with pinned ends. The formula is given by:

\[ P_{cr} = \frac{\pi^2 E I}{(KL)^2} \]

Where:
- \( P_{cr} \) is the critical buckling load.
- \( E \) is the Young's modulus of the material.
- \( I 

## 8. Create RAG Chain with Conversation Memory

In [9]:
# Chat history storage
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Contextualized prompt (includes chat history)
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create history-aware retriever
history_aware_retriever = contextualize_q_prompt | llm | StrOutputParser() | retriever

# QA prompt
qa_system_prompt = """You are an expert engineering tutor for The Efficient Engineer YouTube channel.
Use the following context to answer the question. If you don't know the answer, say so.

{context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create conversational RAG chain
conversational_rag_chain = (
    {
        "context": history_aware_retriever | format_docs,
        "input": RunnablePassthrough(),
        "chat_history": RunnablePassthrough(),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Add message history
conversational_rag_chain_with_history = RunnableWithMessageHistory(
    conversational_rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("✓ Conversational RAG chain with memory created")

✓ Conversational RAG chain with memory created


## 9. Test Conversational Chain

In [13]:
# Helper function for conversational chat


def chat(question, session_id="default"):
    response = conversational_rag_chain_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}}
    )
    
    return response["answer"]
# Test multi-turn conversation
print("="*60)
print("MULTI-TURN CONVERSATION TEST")
print("="*60 + "\n")

questions = [
    "What is stress in engineering?",
    "What are the different types?",
    "How is it calculated?"
]

for i, q in enumerate(questions, 1):
    print(f"[Turn {i}]")
    print(f"User: {q}")
    
    answer = chat(q, session_id="test_session")
    
    print(f"Assistant: {answer}\n")
    print("-" * 60 + "\n")

MULTI-TURN CONVERSATION TEST

[Turn 1]
User: What is stress in engineering?


ValueError: variable chat_history should be a list of base messages, got {'input': 'What is stress in engineering?', 'chat_history': []} of type <class 'dict'>

## 10. Create Reusable Chatbot Class

In [ ]:
class EngineeringChatbot:
    """
    RAG-based chatbot for engineering questions
    """
    
    def __init__(self, index_name='efficient-engineer'):
        # Load API keys
        load_dotenv()
        
        # Initialize embeddings
        self.embeddings = OpenAIEmbeddings(
            openai_api_key=os.getenv('OPENAI_API_KEY')
        )
        
        # Connect to vector store
        self.vectorstore = PineconeVectorStore(
            index_name=index_name,
            embedding=self.embeddings
        )
        
        # Create retriever
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )
        
        # Initialize LLM
        self.llm = ChatOpenAI(
            model="gpt-3.5-turbo",
            temperature=0.7,
            openai_api_key=os.getenv('OPENAI_API_KEY')
        )
        
        # Create simple RAG chain
        template = """You are an expert engineering tutor for The Efficient Engineer.
        Use the context to answer clearly and accurately.
        
        Context: {context}
        Question: {question}
        
        Answer:"""
        
        prompt = ChatPromptTemplate.from_template(template)
        
        self.rag_chain = (
            {"context": self.retriever | self._format_docs, 
             "question": RunnablePassthrough()}
            | prompt
            | self.llm
            | StrOutputParser()
        )
    
    def _format_docs(self, docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    def ask(self, question: str):
        """Ask a question and get response with sources"""
        # Get documents
        docs = self.retriever.get_relevant_documents(question)
        
        # Get answer
        answer = self.rag_chain.invoke(question)
        
        return {
            'answer': answer,
            'sources': docs
        }

print("✓ EngineeringChatbot class created")

## 11. Test Chatbot Class

In [ ]:
# Create chatbot instance
chatbot = EngineeringChatbot()

# Test it
question = "What is the finite element method?"
response = chatbot.ask(question)

print(f"Q: {question}\n")
print(f"A: {response['answer']}\n")
print(f"Sources:")
for i, source in enumerate(response['sources'], 1):
    print(f"  {i}. {source.metadata['video_title']}")

## 12. Interactive Chat Loop

In [ ]:
def interactive_chat():
    """
    Interactive chat loop
    Type 'quit' to exit
    """
    print("\n" + "="*60)
    print("ENGINEERING CHATBOT - Type 'quit' to exit")
    print("="*60 + "\n")
    
    chatbot = EngineeringChatbot()
    
    while True:
        user_input = input("You: ")
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\nGoodbye!")
            break
        
        if not user_input.strip():
            continue
        
        try:
            response = chatbot.ask(user_input)
            print(f"\nAssistant: {response['answer']}")
            
            if response['sources']:
                print(f"\n📚 Sources:")
                for doc in response['sources']:
                    print(f"  - {doc.metadata['video_title']}")
            print()
        except Exception as e:
            print(f"\nError: {e}\n")

# Run interactive chat
interactive_chat()

## 13. Save Chatbot Class to File

In [ ]:
import os
os.makedirs('src', exist_ok=True)

chatbot_code = '''import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

class EngineeringChatbot:
    def __init__(self, index_name="efficient-engineer"):
        load_dotenv()
        
        self.embeddings = OpenAIEmbeddings(
            openai_api_key=os.getenv("OPENAI_API_KEY")
        )
        
        self.vectorstore = PineconeVectorStore(
            index_name=index_name,
            embedding=self.embeddings
        )
        
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
        
        self.llm = ChatOpenAI(
            model="gpt-3.5-turbo",
            temperature=0.7,
            openai_api_key=os.getenv("OPENAI_API_KEY")
        )
        
        template = """You are an expert engineering tutor for The Efficient Engineer.
        Use the context to answer clearly and accurately.
        
        Context: {context}
        Question: {question}
        
        Answer:"""
        
        prompt = ChatPromptTemplate.from_template(template)
        
        self.rag_chain = (
            {"context": self.retriever | self._format_docs, 
             "question": RunnablePassthrough()}
            | prompt
            | self.llm
            | StrOutputParser()
        )
    
    def _format_docs(self, docs):
        return "\\n\\n".join(doc.page_content for doc in docs)
    
    def ask(self, question: str):
        docs = self.retriever.get_relevant_documents(question)
        answer = self.rag_chain.invoke(question)
        return {"answer": answer, "sources": docs}
'''

with open('src/chatbot.py', 'w') as f:
    f.write(chatbot_code)

print("✓ Chatbot saved to src/chatbot.py")

## 14. Summary

In [ ]:
print("="*60)
print("RAG AGENT COMPLETE!")
print("="*60)
print("\n✓ What you built:")
print("  - Simple RAG chain")
print("  - Conversational RAG with memory")
print("  - Reusable chatbot class")
print("  - Interactive chat interface")
print("\n📝 Next steps:")
print("  1. Test the chatbot thoroughly")
print("  2. Run the Streamlit web app")
print("  3. Deploy to production")
print("\n" + "="*60)